In [ ]:
import pandas as pd
import numpy as np
import ast

import os
import matplotlib.pyplot as plt
import seaborn as sns
from plinder_analysis_utils import DockingAnalysisBase, PoseBustersAnalysis, PropertyAnalysis

import statsmodels.formula.api as smf

In [ ]:
PLINDER_TEST_COLUMNS = [
    "system_id", "ligand_smiles",
    # binary 
    # "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    "system_num_pocket_residues", "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", "ligand_num_neighboring_residues", "ligand_num_interactions",
]
# Create category mapping for visualization
CATEGORY_MAPPING = {
    "ligand_is_covalent": "binary",
    "ligand_is_ion": "binary",
    "ligand_is_cofactor": "binary",
    "ligand_is_artifact": "binary",
    "system_num_protein_chains": "discrete",
    "ligand_num_rot_bonds": "continuous",    
    "ligand_num_hbd": "continuous",
    "ligand_num_hba": "continuous",
    "ligand_num_rings": "continuous",
    "entry_resolution": "continuous",
    "entry_validation_molprobity": "continuous",
    "system_num_pocket_residues": "continuous",
    "system_num_interactions": "continuous",
    "ligand_molecular_weight": "continuous",
    "ligand_crippen_clogp": "continuous",
    "ligand_num_interacting_residues": "continuous",
    "ligand_num_neighboring_residues": "continuous",
    "ligand_num_interactions": "continuous",
    "ligand_is_artifact": "binary"     
}

In [ ]:
df_combined = pd.read_csv("plinder_set_0_annotated.csv")

In [ ]:
# build a boolean mask: drop any row where covalent, ionic or has_ion is True
# mask = ~(
#     df_combined['ligand_is_covalent'] |
#     df_combined['ligand_is_ion'] |
#     df_combined['has_ion'] |
#     df_combined['ligand_is_cofactor']
# )

# # filter and reset index
# df_combined = df_combined.loc[mask].reset_index(drop=True)
print("Filtered shape:", df_combined.shape)

In [ ]:
# First analyze multiple properties
property_analysis = PropertyAnalysis(df_combined)
methods = ["surfdock", "gnina", "chai-1", "diffdock_pocket_only", "icm", "vina"]

## Complementarity Between Physics-based and ML-based: Mixed Effect Analysis

### Prepare the df

In [ ]:
MIXED_EFFECT_VARS = [
    "protein", "rmsd","method",
    # "system_id", "ligand_smiles",
    # binary 
    # "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    # "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    # "system_num_pocket_residues", 
    "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", 
    "ligand_num_neighboring_residues", 
    # "ligand_num_interactions",
]

In [ ]:
df_mixed = df_combined[MIXED_EFFECT_VARS]
# Create a Method_Type column based on the classification
df_mixed['Method_Type'] = df_mixed['method'].apply(
    lambda x: 'ML' if x in ['chai-1', 'diffdock_pocket_only', 'surfdock'] else 'Physics'
)

# Display the counts of each method type
print(df_mixed['Method_Type'].value_counts())

# Verify the classification
for method in df_mixed['method'].unique():
    method_type = 'ML' if method in ['chai-1', 'diffdock_pocket_only', 'surfdock'] else 'Physics'
    print(f"{method}: {method_type}")

### system_num_protein_chains

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="system_num_protein_chains",
    bins=[1,2,3,4]
)
print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_protein_chains",
    bins=[1,2,3,4]
)
print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_protein_chains",
    bins=[1,2,3,4]
)
print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_rot_bonds

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rot_bonds",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rot_bonds",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rot_bonds",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_hbd

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hbd",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hbd",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hbd",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

#### ligand_num_hba

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hba",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hba",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hba",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_rings

In [ ]:
df_combined['ligand_num_rings'].value_counts()

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rings",
    bins=[0,3,6,9,12]
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rings",
    bins=[0,3,6,9,12]
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rings",
    bins=[0,3,6,9,12]
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### entry_resolution

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="entry_resolution",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_resolution",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_resolution",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### entry_validation_molprobity

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="entry_validation_molprobity",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_validation_molprobity",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_validation_molprobity",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### system_pocket_residues

### system_num_interactions

### ligand_molecular_weight

### ligand_crippen_clogp

### ligand_num_interactions

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="system_num_interactions",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_interactions",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_interactions",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_neighboring_residues

### ligand_num_interacting_residues

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_interacting_residues",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_interacting_residues",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_interacting_residues",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

## Distribution of the features

In [ ]:
properties = [
    "ligand_num_rot_bonds",
    "ligand_num_hbd",
    "ligand_num_hba",
    "ligand_num_rings",
    "entry_resolution",
    "entry_validation_molprobity",
    "syestem_num_interactions",
    "ligand_molecular_weight",
    "ligand_crippen_clogp",
    "ligand_num_interacting_residues",
    "ligand_num_neighboring_residues",
]

In [ ]:
# First run the complementary analysis
property_analysis = PropertyAnalysis(df_combined)
complementary_results = property_analysis.complementary_success_analysis(
    method1=['diffdock_pocket_only', 'chai-1', 'surfdock'],  # ML methods
    method2=['vina', 'icm'],                   # Physics-based methods
    rmsd_threshold=2.0,
    plot=True
)

# Then analyze property distributions across complementary cases
property_distributions = property_analysis.compare_property_distributions_in_complementary_cases(
    complementary_results=complementary_results,
    properties=properties,
    method_labels=('ML-based', 'Physics-based'),
    property_types={
        'ligand_molecular_weight': 'continuous',
        'ligand_num_rot_bonds': 'continuous',
        'ligand_num_hbd': 'continuous',
        'ligand_num_hba': 'continuous',
        'ligand_num_rings': 'continuous',
        'entry_resolution': 'continuous',
        'entry_validation_molprobity': 'continuous',
        'system_num_interactions': 'continuous',
        'ligand_crippen_clogp': 'continuous',
        'ligand_num_interacting_residues': 'continuous',
        'ligand_num_neighboring_residues': 'continuous'
    },
    plot=True
)

# To view the summary statistics
print(property_distributions['summary_stats']['ligand_molecular_weight'])

# To access the statistical test results
print(property_distributions['test_results']['ligand_num_rot_bonds'])